# MLB Hitting Data (2021–2025)

This notebook pulls **player-level MLB hitting season totals** from the public MLB Stats API and combines seasons **2021, 2022, 2023, 2024, and 2025** into one dataset.


## Data source and API details

- **Source:** MLB Stats API (public)
- **Endpoint:** `https://statsapi.mlb.com/api/v1/stats`
- **Query parameters:**
  - `stats=season` (season totals)
  - `group=hitting` (batting stats)
  - `sportId=1` (MLB)
  - `season` in `2021..2025`
  - `limit` and `offset` for pagination
- **Python dependencies:** `pandas`, `requests`
- **Output file:** `mlb_hitting_2021_2025.csv`

The resulting schema includes identifiers and common batting fields:
`season`, `player_id`, `player_name`, `team_id`, `team_name`, `games_played`, `at_bats`, `runs`, `hits`, `doubles`, `triples`, `home_runs`, `runs_batted_in`, `walks`, `strikeouts`, `stolen_bases`, `caught_stealing`, `batting_average`, `on_base_percentage`, `slugging_percentage`, `ops`.


In [1]:
from pathlib import Path
from typing import Dict, List

import pandas as pd
import requests

API_URL = "https://statsapi.mlb.com/api/v1/stats"
SEASONS = [2021, 2022, 2023, 2024, 2025]
LIMIT = 1000
TIMEOUT = (5, 30)  # connect, read
HEADERS = {
    "User-Agent": "ML-Grad-Project/1.0 (data collection notebook)",
    "Accept": "application/json",
}

OUTPUT_CSV = Path("mlb_hitting_2021_2025.csv")


## Pulling the Data
In this section, we pull the statistics using the API. `fetch_season_hitting_data` takes the argument of the season as a year (ex. 2025).

We limit the pulled hitters to the `playerPool: qualified`. This only pulls qualified hitters, which are players who averaged at least 3.1 plate appearances per game during the regular season. The designation of a qualified hitter is used for season aggregate leaderboards, aka leaders in batting average, on-base percentage, slugging, etc--it is a means of making sure a player with 3 hits in 3 appearances isn't leading the league in batting average. We filter by qualified hitters for a similar purpose, to eliminate outliers during training and to focus on batters of interest, as we want to eventually predict batters who will be playing consistently.


It is worth noting that players who see more than 500 plate appearances per year are above average batters in the MLB--otherwise they wouldn't be playing as much. To this end, if we decide we want to predict stats for unqualified batters, we would likely have to rethink this filter, as the stats are skewed a bit higher.



In [2]:
def fetch_season_hitting_data(season: int, limit: int = LIMIT) -> List[Dict]:
    offset = 0
    all_splits: List[Dict] = []
    total_splits_expected = None

    while True:
        params = {
            "stats": "season",
            "group": "hitting",
            "sportId": 1,
            "season": season,
            "limit": limit,
            "offset": offset,
            "gameType": "R", ## regular season
            "playerPool": "qualified"
        }

        try:
            response = requests.get(API_URL, params=params, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
        except requests.RequestException as exc:
            raise RuntimeError(
                f"Failed to retrieve MLB hitting data for season {season} at offset {offset}: {exc}"
            ) from exc

        try:
            payload = response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"Received non-JSON response for season {season} at offset {offset}."
            ) from exc

        stats_blocks = payload.get("stats")
        if not isinstance(stats_blocks, list) or not stats_blocks:
            raise RuntimeError(
                f"Unexpected response format for season {season}, offset {offset}: missing 'stats'"
            )

        block = stats_blocks[0]
        splits = block.get("splits")
        if splits is None:
            raise RuntimeError(
                f"Unexpected response format for season {season}, offset {offset}: missing 'splits'"
            )

        if total_splits_expected is None:
            total_splits_expected = int(block.get("totalSplits", 0))
            if total_splits_expected <= 0:
                raise RuntimeError(
                    f"No hitting data reported by the API for season {season}; aborting."
                )

        all_splits.extend(splits)

        if len(all_splits) >= total_splits_expected:
            break

        if not splits:
            raise RuntimeError(
                f"Pagination ended early for season {season}. "
                f"Expected {total_splits_expected} rows, got {len(all_splits)}."
            )

        offset += len(splits)

    if total_splits_expected is not None and len(all_splits) != total_splits_expected:
        raise RuntimeError(
            f"Incomplete data for season {season}. "
            f"Expected {total_splits_expected} rows, got {len(all_splits)}."
        )

    return all_splits


def normalize_splits_to_rows(splits: List[Dict], season: int) -> List[Dict]:
    rows = []
    for split in splits:
        player = split.get("player", {})
        team = split.get("team", {})
        stat = split.get("stat", {})

        rows.append(
            {
                "season": season,
                "player_id": player.get("id"),
                "player_name": player.get("fullName"),
                "team_id": team.get("id"),
                "team_name": team.get("name"),
                "games_played": stat.get("gamesPlayed"),
                "at_bats": stat.get("atBats"),
                "runs": stat.get("runs"),
                "hits": stat.get("hits"),
                "doubles": stat.get("doubles"),
                "triples": stat.get("triples"),
                "home_runs": stat.get("homeRuns"),
                "runs_batted_in": stat.get("rbi"),
                "walks": stat.get("baseOnBalls"),
                "strikeouts": stat.get("strikeOuts"),
                "stolen_bases": stat.get("stolenBases"),
                "caught_stealing": stat.get("caughtStealing"),
                "batting_average": stat.get("avg"),
                "on_base_percentage": stat.get("obp"),
                "slugging_percentage": stat.get("slg"),
                "ops": stat.get("ops"),
            }
        )
    return rows

In [3]:
all_rows = []

for season in SEASONS:
    splits = fetch_season_hitting_data(season)
    if not splits:
        raise RuntimeError(f"No hitting rows returned for season {season}; aborting.")
    all_rows.extend(normalize_splits_to_rows(splits, season))

if not all_rows:
    raise RuntimeError("No rows were returned from the MLB Stats API; aborting CSV export.")

hitting_df = pd.DataFrame(all_rows)

column_order = [
    "season",
    "player_id",
    "player_name",
    "team_id",
    "team_name",
    "games_played",
    "at_bats",
    "runs",
    "hits",
    "doubles",
    "triples",
    "home_runs",
    "runs_batted_in",
    "walks",
    "strikeouts",
    "stolen_bases",
    "caught_stealing",
    "batting_average",
    "on_base_percentage",
    "slugging_percentage",
    "ops",
]

hitting_df = hitting_df[column_order]

numeric_columns = [
    "season",
    "player_id",
    "team_id",
    "games_played",
    "at_bats",
    "runs",
    "hits",
    "doubles",
    "triples",
    "home_runs",
    "runs_batted_in",
    "walks",
    "strikeouts",
    "stolen_bases",
    "caught_stealing",
]

for col in numeric_columns:
    hitting_df[col] = pd.to_numeric(hitting_df[col], errors="coerce")

hitting_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(hitting_df):,} rows to {OUTPUT_CSV.resolve()}")


Saved 670 rows to /content/mlb_hitting_2021_2025.csv


In [ ]:
row_counts = hitting_df.groupby("season", dropna=False).size().rename("row_count").reset_index()
row_counts


,season,row_count
0,2021,132
1,2022,130
2,2023,134
3,2024,129
4,2025,145


In [ ]:
hitting_df.head(10)


,season,player_id,player_name,team_id,team_name,games_played,at_bats,runs,hits,doubles,...,home_runs,runs_batted_in,walks,strikeouts,stolen_bases,caught_stealing,batting_average,on_base_percentage,slugging_percentage,ops
0,2021,607208,Trea Turner,119,Los Angeles Dodgers,148,595,107,195,34,...,28,77,41,110,32,5,.328,.375,.536,.911
1,2021,493329,Yuli Gurriel,117,Houston Astros,143,530,83,169,31,...,15,81,59,68,1,1,.319,.383,.462,.845
2,2021,665742,Juan Soto,120,Washington Nationals,151,502,111,157,20,...,29,95,145,93,9,7,.313,.465,.534,.999
3,2021,488726,Michael Brantley,117,Houston Astros,121,469,68,146,29,...,8,47,33,53,1,0,.311,.362,.437,.799
4,2021,665489,Vladimir Guerrero Jr.,141,Toronto Blue Jays,161,604,123,188,29,...,48,111,86,110,4,1,.311,.401,.601,1.002
5,2021,516782,Starling Marte,133,Oakland Athletics,120,467,89,145,27,...,12,55,43,99,47,5,.310,.383,.458,.841
6,2021,547180,Bryce Harper,143,Philadelphia Phillies,141,488,101,151,42,...,35,84,100,134,13,3,.309,.429,.615,1.044
7,2021,641313,Tim Anderson,145,Chicago White Sox,123,527,94,163,29,...,17,61,22,119,18,7,.309,.338,.469,.807
8,2021,592206,Nick Castellanos,113,Cincinnati Reds,138,531,95,164,38,...,34,100,41,121,3,1,.309,.362,.576,.938
9,2021,624428,Adam Frazier,135,San Diego Padres,155,577,83,176,36,...,5,43,48,69,10,5,.305,.368,.411,.779
